# V6 Step 6: Multi-seed Random Projection and No-RP Baseline

This notebook creates V6 Random Projection outputs from the fixed V5 feature matrices only. It does not rerun feature extraction, ResNet1D training, RanPAC, EER, or any Step 7/8 evaluation.


In [2]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

BASE_DIR = Path(r"F:\ECG")
FEATURE_DIR = BASE_DIR / "data" / "processed" / "feature_outputs_v5"
RP_DIR = BASE_DIR / "data" / "processed" / "rp_v6"
SCALER_DIR = RP_DIR / "scaler"
SUMMARY_PATH = RP_DIR / "rp_v6_summary.csv"
NOTEBOOK_PATH = BASE_DIR / "notebooks" / "06_v6_rp.ipynb"

SEEDS = [27, 42, 82]
RP_METHODS = ["gaussian", "sparse", "sign", "srht", "bernoulli_gaussian"]
PROJECTION_DIMS = [100, 200, 500]
EXPECTED_RP_ROWS = 3 * 5 * 3 * 3
EXPECTED_NORP_ROWS = 3
TOTAL_EXPECTED_ROWS = EXPECTED_RP_ROWS + EXPECTED_NORP_ROWS

FEATURE_SETTINGS = {
    "fiducial_pqrst": {
        "alias": "fid",
        "path": FEATURE_DIR / "X_features_fiducial_pqrst.npy",
        "expected_shape": (36103, 23),
    },
    "embedding_resnet1d": {
        "alias": "emb",
        "path": FEATURE_DIR / "X_features_embedding_resnet1d.npy",
        "expected_shape": (36103, 128),
    },
    "both_pqrst_resnet1d": {
        "alias": "both",
        "path": FEATURE_DIR / "X_features_both_pqrst_resnet1d.npy",
        "expected_shape": (36103, 151),
    },
}

RP_ALIASES = {
    "gaussian": "gauss",
    "sparse": "sparse",
    "sign": "sign",
    "srht": "srht",
    "bernoulli_gaussian": "bg",
    "none": "norp",
}

Y_PATH = FEATURE_DIR / "y_stream_feature_ablation_v5.npy"
FEATURE_SUMMARY_PATH = FEATURE_DIR / "feature_ablation_v5_summary.csv"

for folder in [RP_DIR, SCALER_DIR, RP_DIR / "norp"] + [RP_DIR / f"seed{seed}" for seed in SEEDS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Notebook path =", NOTEBOOK_PATH)
print("V6 RP output folder =", RP_DIR)


Notebook path = F:\ECG\notebooks\06_v6_rp.ipynb
V6 RP output folder = F:\ECG\data\processed\rp_v6


## 1. Validate fixed V5 feature inputs

Step 6 V6 uses the existing V5 feature outputs as fixed inputs. This cell checks paths, shapes, labels, and NaN/Inf status before any V6 output is generated.


In [5]:
def stop_if_false(condition, message):
    if not condition:
        raise RuntimeError(message)


stop_if_false(Y_PATH.exists(), f"Missing y stream: {Y_PATH}")
stop_if_false(FEATURE_SUMMARY_PATH.exists(), f"Missing feature summary: {FEATURE_SUMMARY_PATH}")

y_stream = np.load(Y_PATH, mmap_mode="r")
stop_if_false(tuple(y_stream.shape) == (36103,), f"Unexpected y_stream shape: {tuple(y_stream.shape)}")

feature_shapes = {}
for feature_setting, info in FEATURE_SETTINGS.items():
    stop_if_false(info["path"].exists(), f"Missing feature file: {info['path']}")
    X_check = np.load(info["path"], mmap_mode="r")
    stop_if_false(tuple(X_check.shape) == info["expected_shape"], f"Unexpected shape for {feature_setting}: {tuple(X_check.shape)}")
    stop_if_false(not bool(np.isnan(X_check).any()), f"NaN detected in {feature_setting}")
    stop_if_false(not bool(np.isinf(X_check).any()), f"Inf detected in {feature_setting}")
    feature_shapes[feature_setting] = tuple(X_check.shape)

feature_summary = pd.read_csv(FEATURE_SUMMARY_PATH)
print("Feature shapes =", feature_shapes)
print("y_stream shape =", tuple(y_stream.shape))
print("feature_ablation_v5_summary.csv rows =", len(feature_summary))


Feature shapes = {'fiducial_pqrst': (36103, 23), 'embedding_resnet1d': (36103, 128), 'both_pqrst_resnet1d': (36103, 151)}
y_stream shape = (36103,)
feature_ablation_v5_summary.csv rows = 3


## 2. Fit one StandardScaler per feature setting

Each feature setting is standardized independently. The same scaler is reused for all seeds and all RP methods belonging to that feature setting. No-RP baselines also use these standardized features directly.


In [8]:
def fit_manual_standard_scaler(X):
    X = np.asarray(X, dtype=np.float32)
    mean = X.mean(axis=0, dtype=np.float64)
    std = X.std(axis=0, dtype=np.float64)
    scale = std.copy()
    scale[scale == 0.0] = 1.0
    X_scaled = ((X - mean.astype(np.float32)) / scale.astype(np.float32)).astype(np.float32)
    return X_scaled, mean, scale


scaled_features = {}
scaler_paths = {}

for feature_setting, info in FEATURE_SETTINGS.items():
    alias = info["alias"]
    X = np.load(info["path"])
    X_scaled, mean, scale = fit_manual_standard_scaler(X)

    mean_path = SCALER_DIR / f"scaler_{alias}_mean.npy"
    scale_path = SCALER_DIR / f"scaler_{alias}_scale.npy"
    np.save(mean_path, mean)
    np.save(scale_path, scale)

    scaled_features[feature_setting] = X_scaled
    scaler_paths[feature_setting] = {
        "mean": mean_path,
        "scale": scale_path,
    }

    print(feature_setting, "scaled shape =", X_scaled.shape, "mean path =", mean_path, "scale path =", scale_path)


fiducial_pqrst scaled shape = (36103, 23) mean path = F:\ECG\data\processed\rp_v6\scaler\scaler_fid_mean.npy scale path = F:\ECG\data\processed\rp_v6\scaler\scaler_fid_scale.npy
embedding_resnet1d scaled shape = (36103, 128) mean path = F:\ECG\data\processed\rp_v6\scaler\scaler_emb_mean.npy scale path = F:\ECG\data\processed\rp_v6\scaler\scaler_emb_scale.npy
both_pqrst_resnet1d scaled shape = (36103, 151) mean path = F:\ECG\data\processed\rp_v6\scaler\scaler_both_mean.npy scale path = F:\ECG\data\processed\rp_v6\scaler\scaler_both_scale.npy


## 3. Define V5-compatible RP methods

The RP implementations follow the V5 Step 6 rules: Gaussian, sparse Achlioptas-style, sign binarized RP, SRHT without saving a full Hadamard matrix, and Bernoulli-Gaussian RP.


In [11]:
def next_power_of_two(value: int) -> int:
    return 1 << (int(value) - 1).bit_length()


def normalized_fwht_inplace(matrix: np.ndarray) -> np.ndarray:
    n_cols = matrix.shape[1]
    h = 1
    while h < n_cols:
        step = h * 2
        for start in range(0, n_cols, step):
            left = matrix[:, start:start + h].copy()
            right = matrix[:, start + h:start + step].copy()
            matrix[:, start:start + h] = left + right
            matrix[:, start + h:start + step] = left - right
        h = step
    matrix /= np.sqrt(n_cols).astype(np.float32)
    return matrix


def projection_note(input_dim: int, projection_dim: int) -> str:
    if projection_dim < input_dim:
        relation = "compression"
    elif projection_dim > input_dim:
        relation = "randomized feature expansion"
    else:
        relation = "same-dimensional randomized transformation"
    return (
        "V6 multi-seed RP output. Projection dimension is treated as a cancellable template "
        f"dimension / randomized transformation dimension. Input dim {input_dim} -> output dim {projection_dim}: {relation}."
    )


def gaussian_rp(X_scaled, projection_dim, rng):
    input_dim = X_scaled.shape[1]
    W = rng.normal(0.0, 1.0 / np.sqrt(projection_dim), size=(input_dim, projection_dim)).astype(np.float32)
    return (X_scaled @ W).astype(np.float32), W


def sparse_rp(X_scaled, projection_dim, rng):
    input_dim = X_scaled.shape[1]
    s = 3.0
    W = np.zeros((input_dim, projection_dim), dtype=np.float32)
    draws = rng.random((input_dim, projection_dim))
    value = np.sqrt(s) / np.sqrt(projection_dim)
    W[draws < (1.0 / (2.0 * s))] = value
    W[(draws >= (1.0 / (2.0 * s))) & (draws < (1.0 / s))] = -value
    return (X_scaled @ W).astype(np.float32), W


def sign_rp(X_scaled, projection_dim, rng):
    input_dim = X_scaled.shape[1]
    W = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=(input_dim, projection_dim)).astype(np.float32)
    W /= np.sqrt(projection_dim).astype(np.float32)
    Z = X_scaled @ W
    X_rp = np.sign(Z).astype(np.float32)
    X_rp[X_rp == 0.0] = 1.0
    return X_rp, W


def srht_rp(X_scaled, projection_dim, rng, seed):
    num_samples, input_dim = X_scaled.shape
    n_hadamard = next_power_of_two(max(input_dim, projection_dim))
    X_pad = np.zeros((num_samples, n_hadamard), dtype=np.float32)
    X_pad[:, :input_dim] = X_scaled
    random_sign_vector = rng.choice(np.array([-1, 1], dtype=np.int8), size=n_hadamard)
    X_pad *= random_sign_vector.astype(np.float32)
    transformed = normalized_fwht_inplace(X_pad)
    selected_indices = rng.choice(n_hadamard, size=projection_dim, replace=False)
    scaling_factor = float(np.sqrt(n_hadamard / projection_dim))
    X_rp = (transformed[:, selected_indices] * scaling_factor).astype(np.float32)
    params = {
        "n_hadamard": np.array(n_hadamard, dtype=np.int64),
        "random_sign_vector": random_sign_vector,
        "selected_indices": selected_indices.astype(np.int64),
        "scaling_factor": np.array(scaling_factor, dtype=np.float64),
        "seed": np.array(seed, dtype=np.int64),
    }
    return X_rp, params


def bernoulli_gaussian_rp(X_scaled, projection_dim, rng):
    input_dim = X_scaled.shape[1]
    mask_probability = 0.5
    G = rng.normal(0.0, 1.0, size=(input_dim, projection_dim)).astype(np.float32)
    B = rng.binomial(1, mask_probability, size=(input_dim, projection_dim)).astype(np.float32)
    W = (B * G / np.sqrt(mask_probability * projection_dim)).astype(np.float32)
    return (X_scaled @ W).astype(np.float32), W


## 4. Generate 135 RP outputs

This cell generates 3 feature settings x 5 RP methods x 3 dimensions x 3 seeds. Outputs and projection matrices/parameters are saved under the seed-specific V6 folders.


In [14]:
def save_rp_output(feature_setting, X_scaled, method, projection_dim, seed):
    feature_alias = FEATURE_SETTINGS[feature_setting]["alias"]
    rp_alias = RP_ALIASES[method]
    seed_label = f"s{seed}"
    experiment_id = f"{feature_alias}_{rp_alias}_d{projection_dim}_{seed_label}"
    seed_dir = RP_DIR / f"seed{seed}"
    output_path = seed_dir / f"X_{experiment_id}.npy"
    input_dim = X_scaled.shape[1]
    rng = np.random.default_rng(seed)

    if method == "gaussian":
        X_rp, W = gaussian_rp(X_scaled, projection_dim, rng)
        matrix_path = seed_dir / f"W_{experiment_id}.npz"
        np.savez(matrix_path, W=W, seed=np.array(seed, dtype=np.int64), method=method, projection_dim=np.array(projection_dim, dtype=np.int64))
    elif method == "sparse":
        X_rp, W = sparse_rp(X_scaled, projection_dim, rng)
        matrix_path = seed_dir / f"W_{experiment_id}.npz"
        np.savez(matrix_path, W=W, seed=np.array(seed, dtype=np.int64), method=method, projection_dim=np.array(projection_dim, dtype=np.int64), s=np.array(3.0, dtype=np.float64))
    elif method == "sign":
        X_rp, W = sign_rp(X_scaled, projection_dim, rng)
        matrix_path = seed_dir / f"W_{experiment_id}.npz"
        np.savez(matrix_path, W=W, seed=np.array(seed, dtype=np.int64), method=method, projection_dim=np.array(projection_dim, dtype=np.int64))
    elif method == "srht":
        X_rp, params = srht_rp(X_scaled, projection_dim, rng, seed)
        matrix_path = seed_dir / f"SRHT_{feature_alias}_d{projection_dim}_{seed_label}.npz"
        np.savez(matrix_path, **params)
    elif method == "bernoulli_gaussian":
        X_rp, W = bernoulli_gaussian_rp(X_scaled, projection_dim, rng)
        matrix_path = seed_dir / f"W_{experiment_id}.npz"
        np.savez(matrix_path, W=W, seed=np.array(seed, dtype=np.int64), method=method, projection_dim=np.array(projection_dim, dtype=np.int64), mask_probability=np.array(0.5, dtype=np.float64))
    else:
        raise ValueError(f"Unknown RP method: {method}")

    X_rp = X_rp.astype(np.float32, copy=False)
    np.save(output_path, X_rp)

    return {
        "experiment_id": experiment_id,
        "feature_setting_full": feature_setting,
        "feature_alias": feature_alias,
        "input_feature_path": str(FEATURE_SETTINGS[feature_setting]["path"]),
        "input_feature_dim": int(input_dim),
        "rp_method_full": method,
        "rp_alias": rp_alias,
        "projection_dim": int(projection_dim),
        "output_dim": int(X_rp.shape[1]),
        "seed": int(seed),
        "seed_label": seed_label,
        "is_rp": True,
        "is_baseline": False,
        "standardized_before_rp": True,
        "scaler_mean_path": str(scaler_paths[feature_setting]["mean"]),
        "scaler_scale_path": str(scaler_paths[feature_setting]["scale"]),
        "matrix_or_params_path": str(matrix_path),
        "output_path": str(output_path),
        "output_dtype": str(X_rp.dtype),
        "num_samples": int(X_rp.shape[0]),
        "contains_nan": bool(np.isnan(X_rp).any()),
        "contains_inf": bool(np.isinf(X_rp).any()),
        "notes": projection_note(input_dim, projection_dim),
    }


summary_rows = []
start_time = time.time()

for seed in SEEDS:
    for feature_setting in FEATURE_SETTINGS:
        X_scaled = scaled_features[feature_setting]
        for method in RP_METHODS:
            for projection_dim in PROJECTION_DIMS:
                row = save_rp_output(feature_setting, X_scaled, method, projection_dim, seed)
                summary_rows.append(row)
                print(row["experiment_id"])

print("RP rows generated =", len(summary_rows))
print("RP generation seconds =", round(time.time() - start_time, 3))


fid_gauss_d100_s27
fid_gauss_d200_s27
fid_gauss_d500_s27
fid_sparse_d100_s27
fid_sparse_d200_s27
fid_sparse_d500_s27
fid_sign_d100_s27
fid_sign_d200_s27
fid_sign_d500_s27
fid_srht_d100_s27
fid_srht_d200_s27
fid_srht_d500_s27
fid_bg_d100_s27
fid_bg_d200_s27
fid_bg_d500_s27
emb_gauss_d100_s27
emb_gauss_d200_s27
emb_gauss_d500_s27
emb_sparse_d100_s27
emb_sparse_d200_s27
emb_sparse_d500_s27
emb_sign_d100_s27
emb_sign_d200_s27
emb_sign_d500_s27
emb_srht_d100_s27
emb_srht_d200_s27
emb_srht_d500_s27
emb_bg_d100_s27
emb_bg_d200_s27
emb_bg_d500_s27
both_gauss_d100_s27
both_gauss_d200_s27
both_gauss_d500_s27
both_sparse_d100_s27
both_sparse_d200_s27
both_sparse_d500_s27
both_sign_d100_s27
both_sign_d200_s27
both_sign_d500_s27
both_srht_d100_s27
both_srht_d200_s27
both_srht_d500_s27
both_bg_d100_s27
both_bg_d200_s27
both_bg_d500_s27
fid_gauss_d100_s42
fid_gauss_d200_s42
fid_gauss_d500_s42
fid_sparse_d100_s42
fid_sparse_d200_s42
fid_sparse_d500_s42
fid_sign_d100_s42
fid_sign_d200_s42
fid_sign_d500

## 5. Generate 3 no-RP baselines

No-RP is not seed-specific. It saves the standardized feature matrices directly under `rp_v6/norp` and does not save a projection matrix.


In [17]:
def save_norp_output(feature_setting, X_scaled):
    feature_alias = FEATURE_SETTINGS[feature_setting]["alias"]
    input_dim = X_scaled.shape[1]
    experiment_id = f"{feature_alias}_norp"
    output_path = RP_DIR / "norp" / f"X_{experiment_id}.npy"
    X_norp = X_scaled.astype(np.float32, copy=False)
    np.save(output_path, X_norp)

    return {
        "experiment_id": experiment_id,
        "feature_setting_full": feature_setting,
        "feature_alias": feature_alias,
        "input_feature_path": str(FEATURE_SETTINGS[feature_setting]["path"]),
        "input_feature_dim": int(input_dim),
        "rp_method_full": "none",
        "rp_alias": "norp",
        "projection_dim": int(input_dim),
        "output_dim": int(input_dim),
        "seed": 0,
        "seed_label": "norp",
        "is_rp": False,
        "is_baseline": True,
        "standardized_before_rp": True,
        "scaler_mean_path": str(scaler_paths[feature_setting]["mean"]),
        "scaler_scale_path": str(scaler_paths[feature_setting]["scale"]),
        "matrix_or_params_path": "",
        "output_path": str(output_path),
        "output_dtype": str(X_norp.dtype),
        "num_samples": int(X_norp.shape[0]),
        "contains_nan": bool(np.isnan(X_norp).any()),
        "contains_inf": bool(np.isinf(X_norp).any()),
        "notes": "V6 no-RP baseline: standardized feature representation used directly as RanPAC input. This is a reference baseline, not a cancellable biometric transformation.",
    }


for feature_setting in FEATURE_SETTINGS:
    row = save_norp_output(feature_setting, scaled_features[feature_setting])
    summary_rows.append(row)
    print(row["experiment_id"])

print("Total rows after no-RP baselines =", len(summary_rows))


fid_norp
emb_norp
both_norp
Total rows after no-RP baselines = 138


## 6. Save combined V6 summary

One combined summary CSV is saved under `rp_v6/rp_v6_summary.csv` with the required section 25.10 fields.


In [20]:
REQUIRED_SUMMARY_COLUMNS = [
    "experiment_id",
    "feature_setting_full",
    "feature_alias",
    "input_feature_path",
    "input_feature_dim",
    "rp_method_full",
    "rp_alias",
    "projection_dim",
    "output_dim",
    "seed",
    "seed_label",
    "is_rp",
    "is_baseline",
    "standardized_before_rp",
    "scaler_mean_path",
    "scaler_scale_path",
    "matrix_or_params_path",
    "output_path",
    "output_dtype",
    "num_samples",
    "contains_nan",
    "contains_inf",
    "notes",
]

summary_df = pd.DataFrame(summary_rows)[REQUIRED_SUMMARY_COLUMNS]
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("Summary CSV path =", SUMMARY_PATH)
print("Summary rows =", len(summary_df))
print(summary_df.head().to_string(index=False))


Summary CSV path = F:\ECG\data\processed\rp_v6\rp_v6_summary.csv
Summary rows = 138
      experiment_id feature_setting_full feature_alias                                                     input_feature_path  input_feature_dim rp_method_full rp_alias  projection_dim  output_dim  seed seed_label  is_rp  is_baseline  standardized_before_rp                                       scaler_mean_path                                       scaler_scale_path                                        matrix_or_params_path                                                  output_path output_dtype  num_samples  contains_nan  contains_inf                                                                                                                                                                                             notes
 fid_gauss_d100_s27       fiducial_pqrst           fid F:\ECG\data\processed\feature_outputs_v5\X_features_fiducial_pqrst.npy                 23       gaussian    gauss         

## 7. Final validation

The final printout exactly follows the V6 Step 6 validation requirements from AGENT.md section 25.11.


In [23]:
summary_check_df = pd.read_csv(SUMMARY_PATH)

actual_rp_rows = int(summary_check_df["is_rp"].astype(bool).sum())
actual_norp_rows = int(summary_check_df["is_baseline"].astype(bool).sum())
total_actual_rows = len(summary_check_df)

output_files_exist = []
shapes_correct = []
dtype_correct = []
no_nan = []
no_inf = []
sign_binary_checks = []

for _, row in summary_check_df.iterrows():
    output_path = Path(row["output_path"])
    output_files_exist.append(output_path.exists())
    if output_path.exists():
        X_out = np.load(output_path, mmap_mode="r")
        expected_shape = (int(row["num_samples"]), int(row["output_dim"]))
        shapes_correct.append(tuple(X_out.shape) == expected_shape)
        dtype_correct.append(str(X_out.dtype) == "float32")
        no_nan.append(not bool(np.isnan(X_out).any()))
        no_inf.append(not bool(np.isinf(X_out).any()))
        if row["rp_method_full"] == "sign":
            unique_values = np.unique(np.asarray(X_out))
            sign_binary_checks.append(set(unique_values.tolist()).issubset({-1.0, 1.0}))
    else:
        shapes_correct.append(False)
        dtype_correct.append(False)
        no_nan.append(False)
        no_inf.append(False)
        if row["rp_method_full"] == "sign":
            sign_binary_checks.append(False)

rp_matrix_paths = [
    Path(path)
    for path in summary_check_df.loc[summary_check_df["is_rp"].astype(bool), "matrix_or_params_path"].astype(str).tolist()
]
projection_matrices_saved = all(path.exists() for path in rp_matrix_paths)

expected_scaler_files = [
    SCALER_DIR / "scaler_fid_mean.npy",
    SCALER_DIR / "scaler_fid_scale.npy",
    SCALER_DIR / "scaler_emb_mean.npy",
    SCALER_DIR / "scaler_emb_scale.npy",
    SCALER_DIR / "scaler_both_mean.npy",
    SCALER_DIR / "scaler_both_scale.npy",
]
scaler_files_saved = all(path.exists() for path in expected_scaler_files)

final_validation_text = "\n".join([
    f"Expected RP rows = {EXPECTED_RP_ROWS}",
    f"Actual RP rows = {actual_rp_rows}",
    f"Expected no-RP rows = {EXPECTED_NORP_ROWS}",
    f"Actual no-RP rows = {actual_norp_rows}",
    f"Total expected rows = {TOTAL_EXPECTED_ROWS}",
    f"Total actual rows = {total_actual_rows}",
    f"All output files exist = {all(output_files_exist)}",
    f"All shapes correct = {all(shapes_correct)}",
    f"All output dtype float32 = {all(dtype_correct)}",
    f"All no NaN = {all(no_nan)}",
    f"All no Inf = {all(no_inf)}",
    f"Sign RP outputs binary {{-1, +1}} = {all(sign_binary_checks)}",
    f"Projection matrices / SRHT params saved = {projection_matrices_saved}",
    f"Scaler files saved = {scaler_files_saved}",
    f"Summary CSV exists = {SUMMARY_PATH.exists()}",
])

print(final_validation_text)


Expected RP rows = 135
Actual RP rows = 135
Expected no-RP rows = 3
Actual no-RP rows = 3
Total expected rows = 138
Total actual rows = 138
All output files exist = True
All shapes correct = True
All output dtype float32 = True
All no NaN = True
All no Inf = True
Sign RP outputs binary {-1, +1} = True
Projection matrices / SRHT params saved = True
Scaler files saved = True
Summary CSV exists = True
